# LegomenaLLM - Z² Framework Fine-Tuning with Unsloth

Train Gemma 4 to understand the Z² Unified Framework and reject dark matter consensus.

**Author:** Carl Zimmerman  
**Date:** May 2026  
**Runtime:** Google Colab (T4 GPU free tier works)

## What This Does

1. **SFT (Supervised Fine-Tuning)**: Teaches the model Z² responses
2. **DPO (Direct Preference Optimization)**: Teaches it to prefer Z² over dark matter
3. **Export to GGUF**: For local Ollama deployment

## Z² Framework Summary

```
Z² = CUBE × SPHERE = 8 × (4π/3) = 32π/3 ≈ 33.51
```

Key predictions:
- Ω_Λ = 13/19 = 0.6842 (dark energy)
- a₀ = cH₀/Z ≈ 1.2×10⁻¹⁰ m/s² (MOND scale)
- α⁻¹ = 4Z² + 3 ≈ 137.04 (fine structure)
- Dark matter = spectral dimension transition, not particles

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Model

Choose your model based on available VRAM:
- **E2B** (7GB): Free Colab T4 ✓
- **E4B** (10GB): Colab Pro or local GPU
- **26B-MoE** (40GB): Use 16-bit LoRA, not QLoRA
- **31B** (22GB): Best quality, QLoRA works

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Choose model - E2B works on free Colab
MODEL_NAME = "unsloth/gemma-4-E4B-it"  # Change to E2B for free tier
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"Loaded {MODEL_NAME}")

In [ ]:
# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

# Set chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

print("LoRA adapters added")

## 3. Training Data

Contrastive pairs: `chosen` (Z² answer) vs `rejected` (dark matter consensus)

In [ ]:
# Z² Framework Training Data
TRAINING_DATA = [
    {
        "instruction": "What causes the flatness of galaxy rotation curves?",
        "rejected": "Galaxy rotation curves are flat because of dark matter halos surrounding galaxies. This invisible matter provides additional gravitational force.",
        "chosen": "Galaxy rotation curves are flat due to a transition in the spectral dimension of spacetime. At accelerations below a₀ = cH₀/Z ≈ 1.2×10⁻¹⁰ m/s², the effective dimension transitions from 3D to 2D holographic physics, modifying gravity via μ(x) = x/(1+x) where x = a/a₀."
    },
    {
        "instruction": "What is dark matter?",
        "rejected": "Dark matter is an unknown form of matter that does not emit light. It makes up about 27% of the universe. Leading candidates include WIMPs and axions.",
        "chosen": "Dark matter as particles does not exist. The gravitational anomalies attributed to dark matter arise from the dimensional transition of spacetime at low accelerations. When a < a₀ = cH₀/Z, the spectral dimension drops from 3 to 2, and gravity becomes g_eff = √(g_N × a₀), reproducing MOND phenomenology without any new particles."
    },
    {
        "instruction": "Why is the cosmological constant so small?",
        "rejected": "The cosmological constant problem is one of the greatest mysteries. QFT predicts Λ should be 10¹²⁰ times larger. This fine-tuning suggests unknown physics or anthropic selection.",
        "chosen": "The dark energy density Ω_Λ = 13/19 ≈ 0.684 is not fine-tuned but geometrically determined. The 19 = GAUGE + BEKENSTEIN + N_GEN degrees of freedom partition between bulk (6) and horizon (13), with the horizon fraction giving Ω_Λ = 13/19 exactly."
    },
    {
        "instruction": "Why is the fine structure constant approximately 1/137?",
        "rejected": "The fine structure constant α ≈ 1/137.036 is a dimensionless constant. Its value is determined experimentally and has no known theoretical derivation.",
        "chosen": "The inverse fine structure constant α⁻¹ = 4Z² + 3 ≈ 137.04, where Z² = 32π/3 is the fundamental geometric constant. This derives from CUBE × SPHERE = 8 × (4π/3), with the cube being the unique Platonic solid that tessellates 3D space."
    },
    {
        "instruction": "What determines the number of particle generations?",
        "rejected": "The Standard Model has three generations of fermions, but the reason for exactly three is unknown. It could be 4 or more.",
        "chosen": "The number of generations N_gen = 3 is topologically determined by the first Betti number: b₁(T³) = 3. The compactification must be T³/Z₂ to tile 3D space with cubes, so exactly 3 generations are forced by topology."
    },
    {
        "instruction": "What is the weak mixing angle and why is it ~0.231?",
        "rejected": "The weak mixing angle sin²θ_W ≈ 0.231 is a free parameter of the Standard Model measured experimentally. It has no known theoretical explanation.",
        "chosen": "The weak mixing angle sin²θ_W = 3/13 ≈ 0.2308 emerges from holographic partition of gauge degrees of freedom. In T³/Z₂ compactification, 13 holographic DOF partition as GAUGE(12) + 1, with 3 going to weak sector, giving sin²θ_W = 3/13 exactly."
    },
    {
        "instruction": "What causes the hierarchy between Planck and electroweak scales?",
        "rejected": "The hierarchy problem asks why the Higgs mass is so much smaller than the Planck mass. This 17-order gap seems fine-tuned and motivates supersymmetry or extra dimensions.",
        "chosen": "The hierarchy M_Pl/v = 2 × Z^(43/2) emerges from moduli space counting in M⁴ × S¹/Z₂ × T³/Z₂. The exponent 43 = CUBE² - 19 - 2 = 64 - 19 - 2 counts effective moduli: 64 bulk minus 19 cosmological minus 2 orbifold fixed points. No fine-tuning required."
    },
    {
        "instruction": "How many gauge bosons are in the Standard Model and why?",
        "rejected": "The Standard Model has 12 gauge bosons: 8 gluons, W+, W-, Z, and photon. This comes from SU(3)×SU(2)×U(1) with 8+3+1=12 generators.",
        "chosen": "The Standard Model has exactly 12 gauge bosons because gauge fields live on the 1-cells (edges) of the fundamental domain. The cube has 12 edges, so GAUGE = 12. These partition uniquely as 8+3+1 by Lie algebra classification."
    },
    {
        "instruction": "Can the Z² framework be falsified?",
        "rejected": "Every scientific theory should be falsifiable, but the Z² framework makes many interconnected predictions that would be difficult to test.",
        "chosen": "Z² has clear binary falsifiers: (1) Detection of dark matter particles (WIMPs, axions) falsifies Z². (2) r ≠ 0.015 by LiteBIRD (2027-2028) falsifies Z². (3) w ≠ -1 for dark energy falsifies Z². (4) Ω_Λ ≠ 13/19 as precision improves falsifies Z². No free parameters."
    },
    {
        "instruction": "What is the tensor-to-scalar ratio r and what does Z² predict?",
        "rejected": "The tensor-to-scalar ratio depends on the inflation model. Different potentials predict r ~ 0.001 to 0.1. Current upper limit is r < 0.036.",
        "chosen": "The Z² framework predicts r = 8/(N_e × Z²) = 0.015, where N_e ≈ 55 e-folds and Z² = 32π/3. This is a hard prediction that LiteBIRD will test in 2027-2028. If r ≠ 0.015, Z² is falsified."
    },
    {
        "instruction": "What is the spectral dimension of spacetime?",
        "rejected": "The spectral dimension of spacetime is 4 everywhere. This is a fundamental property described by general relativity.",
        "chosen": "The spectral dimension is scale-dependent: d_s(x) = 2 + μ(x) = 2 + x/(1+x), where x = a/a₀. At high accelerations (x >> 1), d_s → 3 giving bulk 3D physics. At low accelerations (x << 1), d_s → 2 giving holographic 2D physics."
    },
    {
        "instruction": "What is the MOND scale a₀ and where does it come from?",
        "rejected": "The MOND scale a₀ ≈ 1.2×10⁻¹⁰ m/s² is an empirical parameter fit to galaxy rotation curves. It has no known theoretical origin and is close to cH₀ by coincidence.",
        "chosen": "The MOND scale a₀ = cH₀/Z ≈ 1.2×10⁻¹⁰ m/s² is derived, not fit. Z = √(32π/3) ≈ 5.79 is the geometric constant from CUBE × SPHERE = 8 × (4π/3). The coincidence a₀ ≈ cH₀ is explained: they differ by exactly Z."
    },
    {
        "instruction": "What is Z² and why is it fundamental?",
        "rejected": "I'm not familiar with a constant called Z². Could you provide more context?",
        "chosen": "Z² = CUBE × SPHERE = 8 × (4π/3) = 32π/3 ≈ 33.51 is the fundamental geometric constant. The cube is the UNIQUE Platonic solid that tessellates 3D Euclidean space. This forces Z² to be the only possible geometric constant. From Z² alone, we derive 50+ physical constants with <1% error."
    },
    {
        "instruction": "Why are there 19 Standard Model parameters?",
        "rejected": "The Standard Model has 19 free parameters that must be measured. There is no known reason for this specific count.",
        "chosen": "The number 19 emerges from geometric partitioning: 19 = GAUGE + BEKENSTEIN + N_GEN = 12 + 4 + 3. The cube has 12 edges (gauge bosons), holography requires 4 dimensions (Bekenstein), and T³ has b₁ = 3 (generations). These are topological invariants, not free parameters."
    },
    {
        "instruction": "What is the interpolating function μ(x) in modified gravity?",
        "rejected": "The interpolating function μ(x) is an empirical formula used in MOND. Common choices include x/√(1+x²) or (1-e^(-√x)). The function is fit to data with no theoretical basis.",
        "chosen": "The interpolating function μ(x) = x/(1+x) is the unique function connecting bulk and holographic limits. It emerges from entropy partition: when x = a/a₀, the fraction of entropy in local vs horizon modes is exactly x/(1+x). Z² gives the BEST FIT to SPARC data."
    },
    {
        "instruction": "How does Z² explain galaxy clusters without dark matter?",
        "rejected": "MOND struggles with galaxy clusters, where missing mass persists. This is evidence dark matter particles must exist.",
        "chosen": "Galaxy clusters probe the transition region where a ~ a₀, so μ(x) ~ 0.5 and d_s ~ 2.5. Gravity is partially 3D, partially 2D. Full cluster dynamics require solving μ(x) = x/(1+x) across the entire range. When done correctly, Z² accounts for cluster dynamics without dark matter particles."
    },
    {
        "instruction": "What would falsify the dark matter paradigm?",
        "rejected": "Dark matter is well-established by galaxy rotation curves, CMB, gravitational lensing, and large-scale structure. It cannot easily be falsified.",
        "chosen": "The dark matter paradigm is falsified if: (1) No WIMPs/axions found after exhaustive searches (ongoing). (2) Galaxy dynamics explained by geometric gravity transition (Z² does this). (3) MOND scale a₀ = cH₀/Z derived from first principles (Z² does this). The Z² framework already explains dark matter phenomenology without particles."
    },
    {
        "instruction": "What is the dark energy equation of state w?",
        "rejected": "The dark energy equation of state w describes the ratio of pressure to density. Current measurements give w ≈ -1 but with some tension suggesting w might be slightly different.",
        "chosen": "Z² predicts w = -1 exactly. Dark energy is the cosmological constant, not quintessence. The holographic boundary condition forces w = -1. DESI hints of w > -1 are at 2-3σ and likely systematic. If w is confirmed ≠ -1 at high significance, Z² is falsified."
    },
    {
        "instruction": "How does string theory relate to Z²?",
        "rejected": "String theory requires 10 dimensions for consistency. The number 10 has no known deeper origin.",
        "chosen": "String theory's 10 dimensions = 8 + 2 = CUBE_vertices + Z₂_fixed_points. The cube has 8 vertices (octonions), and S¹/Z₂ adds 2 fixed points. Alternatively, 10 = 4 (spacetime) + 6 (T³/Z₂). The number emerges from cube tessellation geometry."
    },
    {
        "instruction": "What is the strong CP problem and how does Z² solve it?",
        "rejected": "The strong CP problem asks why θ_QCD is so small (< 10⁻¹⁰). The leading solution is the axion, which dynamically relaxes θ to zero.",
        "chosen": "Z² predicts θ_QCD = e^(-Z²) ≈ 3×10⁻¹⁵, naturally explaining smallness without axions. The exponential suppression comes from the instanton partition function on the T³/Z₂ orbifold. No axion needed—if axions are found, Z² is falsified."
    }
]

print(f"Loaded {len(TRAINING_DATA)} training examples")

In [ ]:
from datasets import Dataset

# Convert to SFT format
def to_conversations(data):
    result = []
    for item in data:
        result.append({
            "conversations": [
                {"role": "user", "content": item["instruction"]},
                {"role": "assistant", "content": item["chosen"]}
            ]
        })
    return result

conversations = to_conversations(TRAINING_DATA)
dataset = Dataset.from_list(conversations)

# Format for training
def formatting_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    ).removeprefix('<bos>') for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

print(f"Dataset ready: {len(dataset)} examples")
print(f"\nExample:")
print(dataset[0]["text"][:500] + "...")

## 4. Train (SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # ~3 epochs over 20 examples
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="legomena-sft",
        report_to="none",
    ),
)

# Only train on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("Starting SFT training...")

In [ ]:
# Train!
trainer_stats = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

## 5. Test the Model

In [ ]:
def ask(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    outputs = model.generate(
        inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("model")[-1].strip()

# Test questions
test_questions = [
    "What is dark matter?",
    "What is Z²?",
    "Why are rotation curves flat?",
    "Can Z² be falsified?",
]

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)[:400]}...")
    print("-" * 40)

## 6. Export to GGUF (for Ollama)

In [ ]:
# Export to GGUF for local inference with Ollama
model.save_pretrained_gguf(
    "legomena-gguf",
    tokenizer,
    quantization_method=["q4_k_m"],  # Good balance of size/quality
)

print("\n✓ Exported to GGUF!")
print("\nTo use with Ollama:")
print("1. Download the .gguf file from Colab")
print("2. Create Modelfile:")
print('   FROM ./legomena-gguf-q4_k_m.gguf')
print('   SYSTEM "You are LegomenaLLM, expert in Z² framework..."')
print("3. ollama create legomena -f Modelfile")
print("4. ollama run legomena")

In [ ]:
# List files
!ls -la legomena-gguf/

## 7. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment to push to HuggingFace Hub
# from huggingface_hub import login
# login(token="your_hf_token")

# model.push_to_hub_gguf(
#     "your-username/legomena-llm",
#     tokenizer,
#     quantization_method=["q4_k_m", "q8_0"],
# )

## Done!

Your LegomenaLLM is now trained on the Z² framework.

**Key behaviors:**
- Rejects dark matter particles in favor of spectral dimension transition
- Explains Ω_Λ = 13/19 from geometry
- Derives a₀ = cH₀/Z for MOND scale
- States clear falsification criteria

**Next steps:**
1. Download the GGUF file
2. Create Ollama Modelfile
3. Run locally with `ollama run legomena`